This workbook allows to list tables with row_count, example of rows from tables and list of expectations metrics (records passed and failed)

In [0]:
# # Show catalog, schemas, tables
 
config_df = spark.sql(""" select s.catalog_name, s.schema_name, t.table_name 
          from system.information_schema.schemata s 
          left join system.information_schema.tables t
          on t.table_catalog = s.catalog_name and t.table_schema = s.schema_name
          where
          s.catalog_name like '%catalog%'
          AND
          s.schema_name in ('bronze','silver','gold')
          AND (t.table_name not like '__materialization%' OR t.table_name is null)
          order by s.catalog_name, s.schema_name, t.table_name
          """)

config_df.display()

In [0]:
# Show tables and their records count
from pyspark.sql.types import StructType, StructField, StringType, LongType

schema = StructType([
    StructField("catalog", StringType(), True),
    StructField("schema", StringType(), True),
    StructField("table", StringType(), True),
    StructField("rows_count", LongType(), True)
])

catalogs = spark.sql("show catalogs").where("catalog ilike '%catalog'").collect()

rows=[]

for c in catalogs:
    schemas= spark.sql(f"show schemas in {c.catalog}").where("databaseName in ('bronze','silver','golden')").collect()
    for s in schemas:
        try:
            tables= spark.sql(f"show tables in {c.catalog}.{s.databaseName}").collect()
        except:
            tables=None
        for t in tables:
            try:
                count = spark.table(f"{c.catalog}.{s.databaseName}.{t.tableName}").count()
            except:
                count = None
            rows.append((c.catalog,s.databaseName,t.tableName, count))

catalogs_schemas_tables_df=spark.createDataFrame(rows,schema)
catalogs_schemas_tables_df.display()

In [0]:
# configuration (choose catalog and provide pipeline id from its settings)
catalog= 'dev_catalog' #use prod_catalog or dev_catalog depending where your run this notebook
schemas = ['bronze','silver','gold']

pipeline_id='03d74c59-b0e3-410d-a1e7-506e9821f9b8'

In [0]:
# show one row from all tables in each of the bronze, silver, golden schema
for schema in schemas:
    tables = [row.tableName for row in spark.sql(f"show tables in {catalog}.{schema}").collect()]
    for table in tables:
        print(f"{catalog}.{schema}.{table}")
        spark.sql(f"select * from {catalog}.{schema}.{table}").limit(1).display()

In [0]:
# show passed_records, failed_records per pipeline run, dataset name and the expectation name 
# data quality metrics are availble for event type = flow_progress and are stored in details:flow_progress:data_quality:expectations array
# expectations is an array of structs with name, dataset, passed_records, failed_records
# origin contains update_id (pipeline run id)

spark.sql(f"""
WITH parsed AS (
    SELECT
        origin.update_id as update_id,
        timestamp,
        parse_json(details):flow_progress:data_quality:expectations AS expectations
    FROM event_log('{pipeline_id}')
    WHERE event_type = 'flow_progress'
)
SELECT
    update_id,
    timestamp,
    e:name              AS expectation_name,
    e:dataset           AS dataset,
    e:passed_records    AS passed_records,
    e:failed_records    AS failed_records
FROM parsed
LATERAL VIEW explode(
    CAST(expectations AS ARRAY<VARIANT>)
) t AS e
""").display()

In [0]:
dataset = "orders_bronze"

spark.sql(f"""
SELECT
    'ingestion' as freshness_type,
    max(ingested_at) as last_event,
    timestampdiff(MINUTE, max(ingested_at), current_timestamp()) as delay_minutes
FROM {catalog}.bronze.{dataset}

UNION ALL

SELECT
    'event',
    max(order_date),
    timestampdiff(MINUTE, max(order_date), current_timestamp())
FROM {catalog}.bronze.{dataset}
""").display()


In [0]:
columns = spark.table(f"{catalog}.bronze.orders_bronze").columns

if "_rescued_data" in columns:
    spark.sql(f"""
        SELECT count(*) AS rescued_rows
        FROM {catalog}.bronze.orders_bronze
        WHERE _rescued_data IS NOT NULL
    """).display()
else:
    print("No schema drift detected")
